In [ ]:
!pip install --upgrade protobuf tensorflow-datasets tensorflow-metadata

In [ ]:
!pip install -q importlib_resources

In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
from keras.applications.resnet50 import ResNet50, preprocess_input

In [2]:
# mount google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# project path
PROJECT_PATH = "/content/drive/MyDrive/project-01-transfer-learning"
os.chdir(PROJECT_PATH)

print("Current Directory: ", os.getcwd())

Current Directory:  /content/drive/MyDrive/project-01-transfer-learning


In [20]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 37

In [10]:
# Load oxford-iiit-pet dataset
(ds_train_raw, ds_tmp_raw), ds_info = tfds.load(
                            'oxford_iiit_pet',
                            split=['train+ test[:40%]', 'test[40%:]'],
                            as_supervised=True,
                            with_info=True
                            )

In [11]:
num_train_data = tf.data.experimental.cardinality(ds_train_raw).numpy()
num_tmp_data = tf.data.experimental.cardinality(ds_tmp_raw).numpy()

print("Number of training data: ", num_train_data)
print("Number of tmp(val+test) data: ", num_tmp_data)

Number of training data:  5148
Number of tmp(val+test) data:  2201


In [13]:
num_val_data = int(0.5 * num_tmp_data)
ds_val_raw = ds_tmp_raw.take(num_val_data)
ds_test_raw = ds_tmp_raw.skip(num_val_data)

num_val_data = tf.data.experimental.cardinality(ds_val_raw).numpy()
num_test_data = tf.data.experimental.cardinality(ds_test_raw).numpy()

print("Number of validation data: ", num_val_data)
print("Number of test data: ", num_test_data)

Number of validation data:  1100
Number of test data:  1101


In [15]:
def resize_and_prep(img, label):
  img = tf.cast(img, tf.float32)
  img = tf.image.resize(img, IMG_SIZE)
  img = preprocess_input(img)

  return img, label

In [16]:
train_prep = ds_train_raw.map(resize_and_prep)
val_prep = ds_val_raw.map(resize_and_prep)
test_prep = ds_test_raw.map(resize_and_prep)

In [18]:
train_ds = train_prep.shuffle(num_train_data).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_prep.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [22]:
# Create a resnemt-50 model
base_model = ResNet50(include_top=False,
                            weights='imagenet',
                            input_shape=(224, 224, 3),
                            )
base_model.trainable = False
base_model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

In [23]:
!pwd

/content/drive/MyDrive/project-01-transfer-learning


In [ ]:
!git co